# 05 - IV/2SLS Main Results

Two-stage least squares estimate of the causal effect of PM2.5 on test scores, using wildfire smoke days as the instrument.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
from linearmodels.iv import IV2SLS

In [ ]:
panel_iv = panel.dropna(subset=["leaid","year","pm25_annual_mean","smoke_days","test_score_mean"]).copy()
panel_iv["year"] = panel_iv["year"].astype(int)

# Demean for within-district / within-year variation (manual FE for linearmodels IV)
for col in ["test_score_mean","pm25_annual_mean","smoke_days"]:
    panel_iv[f"{col}_dm"] = (
        panel_iv[col]
        - panel_iv.groupby("leaid")[col].transform("mean")
        - panel_iv.groupby("year")[col].transform("mean")
        + panel_iv[col].mean()
    )

## 2SLS estimate

In [ ]:
iv = IV2SLS(
    dependent=panel_iv["test_score_mean_dm"],
    exog=None,
    endog=panel_iv[["pm25_annual_mean_dm"]],
    instruments=panel_iv[["smoke_days_dm"]],
).fit(cov_type="robust")

b_iv = iv.params["pm25_annual_mean_dm"]
ci   = iv.conf_int().loc["pm25_annual_mean_dm"]

print(f"IV/2SLS: β(PM2.5) = {b_iv:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
print()
print(f"Interpretation: a 1 μg/m³ increase in annual PM2.5 is associated with")
print(f"  {b_iv:.4f} SD change in test scores (causal estimate via smoke IV)")
print()
print(iv.summary.tables[1])

## Reduced form (smoke -> test scores directly)

In [ ]:
from linearmodels.panel import PanelOLS

idx = panel_iv.set_index(["leaid","year"])
rf = PanelOLS(
    idx["test_score_mean"],
    idx[["smoke_days"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

b_rf = rf.params["smoke_days"]
ci_rf = rf.conf_int().loc["smoke_days"]
print(f"Reduced form: β(smoke_days -> test score) = {b_rf:.4f}  [{ci_rf['lower']:.4f}, {ci_rf['upper']:.4f}]")
print()
print("Reduced form is more credible if exclusion restriction is questioned:")
print("It only requires smoke -> scores, not the two-step PM2.5 channel.")

## OLS vs FE vs IV comparison

In [ ]:
from linearmodels.panel import PanelOLS
import statsmodels.formula.api as smf

m_ols = smf.ols("test_score_mean ~ pm25_annual_mean", data=panel_iv).fit(cov_type="HC3")
fe = PanelOLS(idx["test_score_mean"], idx[["pm25_annual_mean"]],
              entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)

print(f"{'Estimator':<20}  {'β(PM2.5)':>10}  {'Notes'}")
print("-"*60)
print(f"{'Naive OLS':<20}  {m_ols.params['pm25_annual_mean']:>10.4f}  Confounded by poverty")
print(f"{'Panel FE':<20}  {fe.params['pm25_annual_mean']:>10.4f}  Time-invariant confounders removed")
print(f"{'IV/2SLS':<20}  {b_iv:>10.4f}  Causal: only wildfire-driven variation")
print()
print("Expected: IV is more negative than FE (removes remaining endogeneity bias)")